# Session 02 · Advanced Recommender Systems: Types and Implementations

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

In [1]:
# Setup: run from the code/ folder so local modules and data files are found
import os
if os.path.isdir('code'):
    os.chdir('code')

## Example 1 - a synthetic ratings matrix

In [2]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from recsys import factorise, make_ratings, precision_at_k, rmse, split_mask, user_knn_predict

R_true, R, mask = make_ratings(n_users=60, n_items=25, k=3, density=0.3)
print("shape:", R.shape, "observed ratings:", int(mask.sum()), f"({mask.mean():.0%})")

shape: (60, 25) observed ratings: 458 (31%)


## Example 2 - popularity baseline

In [3]:
avg = np.nanmean(np.where(mask, R, np.nan), axis=0)
counts = mask.sum(0)
score = np.where(counts >= 5, avg, 0)
print("top-5 popular items:", np.argsort(-score)[:5], "avg:", np.sort(score)[::-1][:5].round(2))

top-5 popular items: [22  3 18  1 24] avg: [3.67 3.4  3.24 3.13 3.12]


## Example 3 - content-based with TF-IDF

In [4]:
descriptions = ["family drama joint family wedding", "romance love story college", "thriller crime police karachi",
                "family wedding comedy", "historical drama mughal empire", "romance wedding family"]
tfidf = TfidfVectorizer().fit_transform(descriptions)
sim = cosine_similarity(tfidf)
print("items most like item 0:", np.argsort(-sim[0])[1:4], sim[0].round(2))

items most like item 0: [5 3 4] [1.   0.   0.   0.51 0.17 0.56]


## Example 4 - user-based collaborative filtering

In [5]:
train, test = split_mask(mask)
R_train = np.where(train, R, 0)
u, i = np.argwhere(test)[0]
print(f"user {u}, item {i}: predicted {user_knn_predict(R_train, train, u, i):.2f}, actual {R[u, i]:.0f}")

user 0, item 11: predicted 1.83, actual 3


## Example 5 - matrix factorisation by gradient descent

In [6]:
P, Q, B = factorise(R_train, train, k=3, epochs=300)
R_hat = B + P @ Q.T
print("P:", P.shape, "Q:", Q.shape)

P: (60, 3) Q: (25, 3)


## Example 6 - evaluate: RMSE and precision@k

In [7]:
baseline = np.full_like(R, R_train[train].mean(), dtype=float)
print("RMSE global mean:", round(rmse(R, baseline, test), 3))
print("RMSE matrix factorisation:", round(rmse(R, R_hat, test), 3))
print("precision@5 MF:", round(precision_at_k(R_hat, R_true, train, 5), 3),
      "| popularity:", round(precision_at_k(np.tile(score, (R.shape[0], 1)), R_true, train, 5), 3))

RMSE global mean: 1.052
RMSE matrix factorisation: 0.889
precision@5 MF: 0.577 | popularity: 0.323
